# PFF FC Raw Data — Preprocessing Pipeline

**Question:** Can `turf.io.pff.load_match` correctly parse the PFF FC raw format into the
wide-format tracking DataFrame and tidy event DataFrame that `turf.efi.*` expects?

**Data:** FIFA WC 2022 — Brazil vs South Korea (match 10507)  
**Module under test:** `turf.io.pff`

> **Requires:** `data/raw/FIFA_WC_2022/` populated (see project README).  
> Run with `uv sync --group viz` for plots.

In [ ]:
import pathlib

import matplotlib
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

from turf.io.pff import load_match
from turf.efi.line import analyze_lines, fix_player_assignments, vote_line_count
from turf.efi.viz import make_pitch, plot_lines_frame

%matplotlib inline
matplotlib.rcParams['figure.facecolor'] = '#ffffff'

np.random.seed(42)

RAW_DIR     = pathlib.Path('../../data/raw/FIFA_WC_2022')
RESULTS_DIR = pathlib.Path('results')
RESULTS_DIR.mkdir(exist_ok=True)

MATCH_ID = 10507

## Load raw data

In [ ]:
match = load_match(RAW_DIR, MATCH_ID)

print(f'Home team : {match.meta["homeTeam"]["name"]} ({match.meta["homeTeam"]["shortName"]})')
print(f'Away team : {match.meta["awayTeam"]["name"]} ({match.meta["awayTeam"]["shortName"]})')
print(f'FPS       : {match.meta["fps"]}')
print()
print(f'Tracking  : {match.tracking.shape[0]:,} frames × {match.tracking.shape[1]} cols')
print(f'Events    : {match.events.shape[0]:,} rows × {match.events.shape[1]} cols')
print()
print('Attacking directions (from homeTeamStartLeft metadata):')
print(f'  Period 1 — Home: {match.directions.home_p1:+d}  Away: {match.directions.away_p1:+d}')
print(f'  Period 2 — Home: {match.directions.home_p2:+d}  Away: {match.directions.away_p2:+d}')

## Tracking DataFrame structure

In [ ]:
tr = match.tracking

print('Columns (first 12):', list(tr.columns[:12]))
print()

player_x_cols = [c for c in tr.columns if c.endswith('_x') and not c.startswith('ball')]
home_players = sorted({c.split('_')[1] for c in player_x_cols if c.startswith('Home')})
away_players = sorted({c.split('_')[1] for c in player_x_cols if c.startswith('Away')})
print(f'Home jersey numbers ({len(home_players)}): {home_players}')
print(f'Away jersey numbers ({len(away_players)}): {away_players}')
print()

periods = tr['Period'].value_counts().sort_index()
print('Frames per period:')
for p, n in periods.items():
    print(f'  Period {p}: {n:,}')
print()

print('ball_x NaN rate:', tr['ball_x'].isna().mean().round(4))
print()
print(tr[['Period', 'Time [s]', 'ball_x', 'ball_y', 'Home_9_x', 'Away_1_x']].head(5).to_string())

## Verify attacking directions

At kick-off (first frame), the GK should be at the extreme x of their own goal.
- BRA (Home) starts left → GK at min x (most negative) → `home_p1 = +1`
- KOR (Away) starts right → GK at max x (most positive) → `away_p1 = -1`

In [ ]:
first_frame = tr[tr['Period'] == 1].iloc[0]

home_x = {c.split('_')[1]: first_frame[c] for c in tr.columns
           if c.startswith('Home_') and c.endswith('_x') and not pd.isna(first_frame[c])}
away_x = {c.split('_')[1]: first_frame[c] for c in tr.columns
           if c.startswith('Away_') and c.endswith('_x') and not pd.isna(first_frame[c])}

home_gk_num = min(home_x, key=lambda k: home_x[k])  # most negative = defending left
away_gk_num = max(away_x, key=lambda k: away_x[k])  # most positive = defending right

print(f'Home GK at kick-off: jersey #{home_gk_num}  x={home_x[home_gk_num]:.2f}')
print(f'Away GK at kick-off: jersey #{away_gk_num}  x={away_x[away_gk_num]:.2f}')
print()
print(f'Metadata says home_p1={match.directions.home_p1:+d}')
print(f'  → Home GK should be at min x: {match.directions.home_p1 == 1}')
print(f'Metadata says away_p1={match.directions.away_p1:+d}')
print(f'  → Away GK should be at max x: {match.directions.away_p1 == -1}')

## Event DataFrame structure

In [ ]:
ev = match.events

print('Columns:', list(ev.columns))
print()
print('possession_type distribution:')
print(ev['possession_type'].value_counts().head(8).to_string())
print()
print('lines_broken distribution (PA events only):')
passes = ev[ev['possession_type'] == 'PA']
print(passes['lines_broken'].value_counts(dropna=False).to_string())
print()
line_breaking = passes[passes['lines_broken'].notna() & (passes['lines_broken'] != 'NONE')]
pct = len(line_breaking) / len(passes) * 100
print(f'Line-breaking passes: {len(line_breaking)} / {len(passes)} ({pct:.1f}%)')

## End-to-end: clip → analyze_lines using metadata direction

The metadata gives attacking direction directly — no need to call `infer_attacking_direction`.
Use `match.directions.for_team(team, period)` wherever `analyze_lines` is called.

In [ ]:
# Pick a P1 BRA pass where KOR (Away) is defending
bra_passes = ev[
    (ev['possession_type'] == 'PA') &
    (ev['is_home'] == True) &
    (ev['period'] == 1) &
    ((ev['end_time'] - ev['start_time']) >= 0.4)
].reset_index(drop=True)

event_row = bra_passes.iloc[20]
period     = int(event_row['period'])
start_t    = float(event_row['start_time'])
end_t      = float(event_row['end_time'])
defending  = 'Away'    # KOR defending
attacking  = 'Home'    # BRA attacking

clip = tr[
    (tr['Period'] == period) &
    (tr['Time [s]'] >= start_t) &
    (tr['Time [s]'] <= end_t)
].copy().reset_index(drop=True)

# Direction from metadata — no inference needed
atk_dir = match.directions.for_team(defending, period)

first_pass = analyze_lines(clip, defending, atk_dir)
k = vote_line_count(first_pass)
lines_df = fix_player_assignments(
    analyze_lines(clip, defending, atk_dir, force_n_lines=k),
    defending,
)

print(f'Event: {event_row["passer_name"]} → {event_row["receiver_name"]}')
print(f'Time window: {start_t:.1f}s – {end_t:.1f}s  ({len(clip)} frames)')
print(f'Defending: {defending} (KOR)  attacking_direction = {atk_dir:+d}')
print(f'Voted K = {k}')
print()

first_row = lines_df.iloc[0]
assignments: dict[int, list[str]] = {}
n = 1
while True:
    xc = f'{defending}_{n}_x'
    if xc not in first_row.index:
        break
    lc = f'{defending}_{n}_line'
    import math
    try:
        ln = int(float(first_row[lc]))
        if not math.isnan(float(first_row[xc])):
            assignments.setdefault(ln, []).append(str(n))
    except (KeyError, ValueError, TypeError):
        pass
    n += 1

for ln_key in sorted(assignments):
    label = 'GK' if ln_key == 0 else f'Line {ln_key}'
    print(f'  {label}: players {assignments[ln_key]}')

## Visualise

In [ ]:
pitch, theme = make_pitch()
fig, ax = pitch.draw(figsize=(12, 8))
fig.patch.set_facecolor(theme['bg_color'])
ax.set_ylim(-42, 34)

plot_lines_frame(
    ax, lines_df.iloc[0],
    defending_team=defending, attacking_team=attacking,
    theme=theme,
)

ax.set_title(
    f'{defending} (KOR) unit lines — K={k} — from raw PFF data\n'
    f'attacking_direction from metadata homeTeamStartLeft (no inference)',
    color=theme['text_color'], fontsize=11, pad=8
)
plt.tight_layout()
plt.savefig(RESULTS_DIR / 'pipeline_check.png', dpi=120, bbox_inches='tight')
plt.show()

## Findings

- `load_match` correctly parses the PFF FC JSONL.bz2 tracking format into the
  wide DataFrame that `turf.efi.line` expects.
- `homeTeamStartLeft` in metadata gives attacking direction directly — more reliable
  than inferring from tracking clips (no edge cases around corners, full press, etc.).
- `lines_broken` in the event DataFrame maps directly to PFF's `linesBrokenType`
  (values: `None`, `'NONE'`, `'MD'`, `'ATT'`, …) — this replaces the separate
  `lbp_annotation_pff_wc22.csv` file used in earlier experiments.
- **Next:** build `turf.io.pff.save_match` to write processed data to parquet,
  then update all experiment notebooks to load from raw rather than from the
  preprocessed CSV files in `data/pff/wc2022/`.